In [1]:
%pip install -q "python-telegram-bot>=22,<23" httpx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.4/769.4 kB 33.1 MB/s eta 0:00:00


In [7]:
import asyncio
import logging

import httpx
from google.colab import userdata
from telegram.ext import (
    Application,
    CommandHandler,
    MessageHandler,
    filters,
)

# جلوگیری از نمایش آدرس درخواست‌ها و کلیدها در لاگ
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

# کلیدها فقط از Secrets خوانده می‌شوند
TELEGRAM_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN").strip()
WEATHER_KEY = userdata.get("WEATHER_API_KEY").strip()

if not TELEGRAM_TOKEN or not WEATHER_KEY:
    raise ValueError("Please fill in both Colab Secrets.")


# برای راحتی تست با چند نام فارسی
CITY_ALIASES = {
    "رشت": "Rasht,IR",
    "تهران": "Tehran,IR",
    "شیراز": "Shiraz,IR",
    "اصفهان": "Isfahan,IR",
    "مشهد": "Mashhad,IR",
    "تبریز": "Tabriz,IR",
    "انزلی": "Bandar Anzali,IR",
}


async def start(update, context):
    await update.effective_message.reply_text(
        "سلام! اسم شهر رو بفرست تا دمای فعلیش رو بگم.\n"
        "مثلاً: رشت یا Rasht,IR\n"
        "برای شهرهای هم‌نام، کد کشور رو هم اضافه کن."
    )


from datetime import datetime, timedelta, timezone


def api_error_message(status):
    if status == 404:
        return "شهر پیدا نشد؛ اسم انگلیسی و کد کشور رو امتحان کن."

    if status in (401, 403):
        return "کلید API یا دسترسی حساب به این سرویس تأیید نشد."

    if status == 429:
        return "محدودیت درخواست سرویس پر شده؛ کمی بعد امتحان کن."

    return "سرویس هواشناسی فعلاً پاسخ مناسبی نداد."


def summarize_forecast(current, forecast):
    # تاریخ‌ها با اختلاف ساعت شهر محاسبه می‌شوند، نه ساعت کولب
    city_timezone = timezone(
        timedelta(seconds=int(current["timezone"]))
    )
    now = datetime.now(city_timezone)
    today = now.date()
    tomorrow = today + timedelta(days=1)

    remaining_today = []
    tomorrow_points = []

    for item in forecast["list"]:
        local_time = datetime.fromtimestamp(
            item["dt"], tz=city_timezone
        )
        temperature = float(item["main"]["temp"])

        if local_time.date() == today and local_time > now:
            remaining_today.append(temperature)

        elif local_time.date() == tomorrow:
            tomorrow_points.append((local_time, temperature))

    lines = []

    if remaining_today:
        # دمای فعلی هم در بازهٔ «از الان تا پایان امروز» لحاظ می‌شود
        temperatures = [
            float(current["main"]["temp"]),
            *remaining_today,
        ]

        lines.extend([
            "",
            "📅 از الان تا پایان امروز — تخمینی",
            f"🔻 کمینه: {min(temperatures):.1f} °C",
            f"🔺 بیشینه: {max(temperatures):.1f} °C",
        ])
    else:
        lines.extend([
            "",
            "📅 برای باقی‌ماندهٔ امروز پیش‌بینی دیگری موجود نیست.",
        ])

    if tomorrow_points:
        temperatures = [temp for _, temp in tomorrow_points]

        # نزدیک‌ترین نمونهٔ پیش‌بینی به ساعت ۱۲ ظهر فردا
        noon = datetime.combine(
            tomorrow,
            datetime.min.time(),
            tzinfo=city_timezone,
        ) + timedelta(hours=12)

        noon_time, noon_temperature = min(
            tomorrow_points,
            key=lambda point: abs((point[0] - noon).total_seconds()),
        )

        lines.extend([
            "",
            f"🌤 پیش‌بینی فردا ({tomorrow:%Y-%m-%d})",
            f"🌡 دما در ساعت {noon_time:%H:%M}: "
            f"{noon_temperature:.1f} °C",
            f"🔻 کمینهٔ نمونه‌ها: {min(temperatures):.1f} °C",
            f"🔺 بیشینهٔ نمونه‌ها: {max(temperatures):.1f} °C",
            "",
            "بازه‌ها از نمونه‌های سه‌ساعته محاسبه شده‌اند؛ "
            "کمینه و بیشینهٔ دقیق روز نیستند.",
        ])
    else:
        lines.extend([
            "",
            "پیش‌بینی فردا در پاسخ سرویس موجود نیست.",
        ])

    return "\n".join(lines)


async def weather(update, context):
    message = update.effective_message
    city = (
        message.text.strip()
        .replace("ي", "ی")
        .replace("ك", "ک")
    )

    if not city or len(city) > 100:
        await message.reply_text("لطفاً فقط اسم شهر رو وارد کن.")
        return

    query = CITY_ALIASES.get(city, city)

    try:
        async with httpx.AsyncClient(timeout=15) as client:
            # دریافت دما، رطوبت و موقعیت شهر
            response = await client.get(
                "https://api.openweathermap.org/data/2.5/weather",
                params={
                    "q": query,
                    "appid": WEATHER_KEY,
                    "units": "metric",
                },
            )

            if response.status_code != 200:
                await message.reply_text(
                    api_error_message(response.status_code)
                )
                return

            current = response.json()

            temperature = float(current["main"]["temp"])
            humidity = int(current["main"]["humidity"])
            city_name = current["name"]
            country = current.get("sys", {}).get("country", "")

            location = (
                f"{city_name}, {country}" if country else city_name
            )

            reply = (
                f"📍 {location}\n"
                f"🌡 دمای فعلی: {temperature:.1f} °C\n"
                f"💧 رطوبت فعلی: {humidity}٪"
            )

            # با مختصات همان شهر، پیش‌بینی را دریافت می‌کنیم
            # اگر پیش‌بینی خطا داد، اطلاعات فعلی همچنان ارسال می‌شود
            try:
                forecast_response = await client.get(
                    "https://api.openweathermap.org/data/2.5/forecast",
                    params={
                        "lat": current["coord"]["lat"],
                        "lon": current["coord"]["lon"],
                        "appid": WEATHER_KEY,
                        "units": "metric",
                    },
                )

                if forecast_response.status_code == 200:
                    reply += summarize_forecast(
                        current,
                        forecast_response.json(),
                    )
                else:
                    reply += (
                        "\n\nپیش‌بینی دریافت نشد:\n"
                        + api_error_message(
                            forecast_response.status_code
                        )
                    )

            except httpx.RequestError:
                reply += "\n\nارتباط با سرویس پیش‌بینی برقرار نشد."

            except (ValueError, KeyError, TypeError, OverflowError):
                reply += "\n\nدادهٔ پیش‌بینی ناقص یا قابل خواندن نیست."

    except httpx.RequestError:
        reply = "ارتباط با سرویس هواشناسی برقرار نشد."

    except (ValueError, KeyError, TypeError):
        reply = "پاسخ سرویس هواشناسی قابل خواندن نبود."

    await message.reply_text(reply)


async def handle_error(update, context):
    # متن کامل خطا را چاپ نمی‌کنیم تا کلیدها افشا نشوند.
    print("Bot error:", type(context.error).__name__)


async def run_bot():
    app = Application.builder().token(TELEGRAM_TOKEN).build()

    app.add_handler(CommandHandler("start", start))
    app.add_handler(
        MessageHandler(filters.TEXT & ~filters.COMMAND, weather)
    )
    app.add_error_handler(handle_error)

    try:
        async with app:
            await app.start()

            try:
                await app.updater.start_polling(
                    drop_pending_updates=False,
                    allowed_updates=["message"],
                )

                print("ربات روشن شد؛ در تلگرام /start بفرست.")
                print("برای خاموش‌کردن، دکمهٔ توقف همین سلول را بزن.")

                # ربات تا زمان توقف سلول منتظر پیام می‌ماند
                await asyncio.Event().wait()

            finally:
                if app.updater.running:
                    await app.updater.stop()
                if app.running:
                    await app.stop()

    except asyncio.CancelledError:
        print("ربات متوقف شد.")

    except Exception as error:
        print("ربات اجرا نشد. نوع خطا:", type(error).__name__)


# مناسب محیط کولب؛ نیازی به asyncio.run نیست
await run_bot()

ربات روشن شد؛ در تلگرام /start بفرست.
برای خاموش‌کردن، دکمهٔ توقف همین سلول را بزن.
ربات متوقف شد.
